In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
from matplotlib.patches import Rectangle

PROJECT_DIR = Path.cwd().parent
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from core.config import P, settings
from core.setup import setup_data, setup_logging, setup_project_path
from domain.annotations import load_annotations
from utils.audio import load_clip

setup_logging(settings.LOG_LEVEL)
setup_project_path(PROJECT_DIR)
# setup_data()

CLEANED_DIR = settings.data_dir / "cleaned"
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [ ]:
annotations = load_annotations(CLEANED_DIR)

print(f"{len(annotations)} anotaciones | {annotations.species.nunique()} especies")
annotations.groupby(["species", "call_type"]).size().sort_values(ascending=False)

In [ ]:
LABEL_MAP = {
    "0": "aa/gc",
    "1": "ac/bc",
    "2": "as/bc",
    "3": "as/hc",
    "4": "lw/cs",
    "5": "pt/dc",
    "6": "sb/ppc",
    "7": "sb/spc",
    "8": "sm/cc",
    "9": "sm/fs",
}
CLASS_LABELS = list(LABEL_MAP.values())

annotations["label"] = annotations["species"] + "/" + annotations["call_type"]
class_df = annotations[annotations["label"].isin(CLASS_LABELS)].copy()
class_df["label"] = pd.Categorical(class_df["label"], categories=CLASS_LABELS, ordered=True)

print(f"{len(class_df)} anotaciones en {len(CLASS_LABELS)} clases")
class_df["label"].value_counts().sort_index()

In [ ]:
METRICS = ["duration_s", "low_freq_hz", "high_freq_hz", "bandwidth_hz"]
STATS = ["count", "mean", "std", "min", "median", "max"]

class_stats = class_df.groupby("label", observed=True)[METRICS].agg(STATS)
class_stats.to_csv(settings.data_dir / "class_stats.csv")

In [ ]:
from domain.species import LabelSet
from pipelines.inference_pipeline import predict as run_inference


def _draw_boxes(ax, rows: pd.DataFrame, start: float, params, color: str, with_score: bool) -> None:
    for _, row in rows.iterrows():
        x0 = max(row["Begin Time (s)"] - start, 0.0)
        x1 = min(row["End Time (s)"] - start, params.clip_len_s)
        ax.add_patch(
            Rectangle(
                (x0, row["Low Freq (Hz)"]),
                x1 - x0,
                row["High Freq (Hz)"] - row["Low Freq (Hz)"],
                edgecolor=color,
                facecolor="none",
                linewidth=1.5,
            )
        )
        label = f"{row['Species']}/{row['Call type']}"
        if with_score:
            label += f" {row['Score']:.2f}"
        ax.text(x0, row["High Freq (Hz)"], label, color=color, fontsize=10, fontweight="bold", va="bottom")


def plot_clips_with_boxes(
    wav_path: Path,
    annotations_path: Path,
    score_threshold: float | None = None,
    max_clips: int | None = None,
    model=None,
    labels: LabelSet | None = None,
    device: str = "cpu",
    params=P,
) -> None:
    """Espectrograma lineal (sin mel) en clips de `clip_len_s` sin overlap. Cajas de
    `annotations_path` en cian. Si además se pasan `model` + `labels` (LabelSet), corre
    la inferencia sobre el mismo audio y dibuja sus detecciones en verde -- para
    comparar visualmente predicción vs ground truth."""
    ann = pd.read_csv(annotations_path, sep="\t")
    has_score = "Score" in ann.columns
    if has_score and score_threshold is not None:
        ann = ann[ann["Score"] >= score_threshold]

    preds = None
    if model is not None:
        if labels is None:
            raise ValueError("pasa `labels` (LabelSet) junto con `model`")
        preds = run_inference(
            model, wav_path, labels, device, score_threshold=score_threshold or 0.5, params=params
        )

    duration_s = sf.info(wav_path).duration
    n_freq_bins = params.n_fft // 2 + 1
    freqs_hz = np.linspace(0.0, params.target_sr / 2, n_freq_bins)
    spectrogram = torchaudio.transforms.Spectrogram(
        n_fft=params.n_fft, win_length=params.win_length, hop_length=params.hop_length, power=2.0
    )

    starts = np.arange(0.0, duration_s, params.clip_len_s)
    if max_clips is not None:
        starts = starts[:max_clips]

    for start in starts:
        waveform = load_clip(wav_path, float(start), params)
        spec_db = 10 * torch.log10(spectrogram(waveform) + params.eps)

        _, ax = plt.subplots(figsize=(10, 4))
        ax.imshow(
            spec_db.numpy(),
            origin="lower",
            aspect="auto",
            extent=(0, params.clip_len_s, freqs_hz[0], freqs_hz[-1]),
            cmap="magma",
        )

        clip_ann = ann[
            (ann["End Time (s)"] > start) & (ann["Begin Time (s)"] < start + params.clip_len_s)
        ]
        _draw_boxes(ax, clip_ann, start, params, "cyan", has_score)

        if preds is not None:
            clip_preds = preds[
                (preds["End Time (s)"] > start) & (preds["Begin Time (s)"] < start + params.clip_len_s)
            ]
            _draw_boxes(ax, clip_preds, start, params, "lime", True)
            ax.legend(
                handles=[
                    Rectangle((0, 0), 1, 1, edgecolor="cyan", facecolor="none", label="GT"),
                    Rectangle((0, 0), 1, 1, edgecolor="lime", facecolor="none", label="modelo"),
                ],
                loc="lower right",
                fontsize=9,
            )

        if score_threshold is not None and (has_score or preds is not None):
            ax.text(
                0.02, 0.95, f"thr={score_threshold:.2f}",
                transform=ax.transAxes, fontsize=10, fontweight="bold", color="yellow", va="top",
            )

        ax.set_xlabel("Tiempo (s)")
        ax.set_ylabel("Frecuencia (Hz)")
        ax.set_title(f"{Path(wav_path).name + ' - ' + str(Path(wav_path).parent.name)} @ {start:.1f}s")
        plt.show()


In [ ]:
sample_audio= settings.data_dir / "cleaned" / "weddells_saddleBack_tamarin__LW/240125_0028.wav"
sample_annotations= settings.data_dir / "cleaned" / "weddells_saddleBack_tamarin__LW/240125_0028.selections.txt"

plot_clips_with_boxes(sample_audio, sample_annotations, score_threshold=0.5)

In [ ]:
from domain.dataset import build_manifest, split_manifest
from domain.species import LabelSet
from main import LABEL_BY, LABEL_COLUMN, MIN_PAIR_COUNT, SEED

CHECKPOINT_PATH = PROJECT_DIR / "checkpoints" / "128d_64q_iouiou_10cls_best.pth"
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
device = "cuda" if torch.cuda.is_available() else "cpu"

# mismo filtrado que main.preprocess(), para poder reproducir el split con SEED
raw = annotations[
    ~(
        ((annotations["species"] == "lw") & (annotations["call_type"] == "cc"))
        | ((annotations["species"] == "sm") & (annotations["call_type"] == "fc"))
    )
].copy()
raw.loc[raw["low_freq_hz"] <= 50.0, "low_freq_hz"] = 50.0

pairs = raw[["species", "call_type"]].apply(tuple, axis=1)
valid_pairs = pairs.value_counts()[lambda s: s >= MIN_PAIR_COUNT].index
experiment_df = raw[pairs.isin(valid_pairs)].copy()
experiment_df["label"] = LABEL_COLUMN[LABEL_BY](experiment_df)

labels = LabelSet(experiment_df["label"])
assert labels.names == checkpoint["labels"], "el split no coincide con el checkpoint"

manifest = build_manifest(experiment_df, labels)
train_m, val_m, test_m = split_manifest(manifest, seed=SEED)

for name, split in [("train", train_m), ("val", val_m), ("test", test_m)]:
    files = sorted({w.audio_path for w in split})
    print(f"{name}: {len(files)} archivos, {len(split)} ventanas")
    for f in files:
        print(f"  {f}")

In [ ]:
from architectures.criterion import HungarianMatcher, SetCriterion
from architectures.deformable_detr import ASTDeformableDETR
from domain.dataset import CallBoxDataset
from main import (
    MATCHER_IOU_TYPE,
    METRIC_IOU_THRESHOLD,
    METRIC_IOU_TYPE,
    OPERATING_SCORE_THRESHOLD,
    format_ap,
    format_confusion,
    make_loader,
)
from pipelines.training_pipeline import evaluate

model = ASTDeformableDETR(
    dim=checkpoint["dim"], n_queries=checkpoint["n_queries"], n_classes=len(labels)
).to(device)
model.load_state_dict(checkpoint["state_dict"])

matcher = HungarianMatcher(iou_type=MATCHER_IOU_TYPE)
criterion = SetCriterion(n_classes=len(labels), matcher=matcher, iou_type=MATCHER_IOU_TYPE).to(device)

# val+test combinado: val ya se usó para elegir el checkpoint, test nunca se tocó
eval_loader = make_loader(CallBoxDataset(val_m + test_m), shuffle=False)
metrics = evaluate(
    model,
    eval_loader,
    criterion,
    matcher,
    device,
    n_classes=len(labels),
    iou_threshold=METRIC_IOU_THRESHOLD,
    ap_iou_type=METRIC_IOU_TYPE,
    score_threshold=OPERATING_SCORE_THRESHOLD,
)

print(
    f"val+test ({len(val_m) + len(test_m)} ventanas) -> "
    f"recall_agn@{METRIC_IOU_THRESHOLD}={metrics.recall_agn:.3f} "
    f"FP/TP={metrics.fp_per_tp:.2f} mAP={metrics.map:.3f} "
    f"cls_acc={metrics.cls_acc:.3f} IoU={metrics.mean_iou:.3f}"
)
print("AP por clase ->", format_ap(metrics.ap_per_class, labels.names))
print(format_confusion(metrics.confusion, labels.names))

In [ ]:
sample_files = sorted({w.audio_path for w in val_m + test_m if "LW" in Path(w.audio_path).parent.name})[6:10]
for wav_path in sample_files:
    ann_path = Path(wav_path).with_suffix(".txt")
    plot_clips_with_boxes(
        Path(wav_path), ann_path,
        model=model, labels=labels, device=device, score_threshold=0.8,
    )